### 1 - Data Ingesion & Integration

In [10]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when

In [11]:
#Dynamic Path Detection so it runs on all user environments
BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "data")

In [12]:
#Spark Initialization
spark = SparkSession.builder \
    .appName("BODS_Predictive_Engine") \
    .config("spark.hadoop.fs.defaultFS", "file:///") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

In [13]:
# Load files
df_cat = spark.read.csv(os.path.join(DATA_DIR, "overall_data_catalogue.csv"), header=True, inferSchema=True)
df_comp = spark.read.csv(os.path.join(DATA_DIR, "overall_compliance_report.csv"), header=True, inferSchema=True)

# Surgical Join: Joining on Service Code (Catalogue) and Service Number (Compliance) <Multi Catalogue Ingestion>
df_unified = df_cat.join(
    df_comp, 
    df_cat["Service Code"] == df_comp["Registration:Service Number"], 
    how="right"
)

print(f"Row count after surgical join: {df_unified.count()}")

[Stage 43:=========>        (2 + 2) / 4][Stage 44:>                 (0 + 2) / 2]

Row count after surgical join: 103437


### 2 - Data Cleaning

In [14]:
# 1. Drop rows where critical compliance data is missing
df_cleaned = df_unified.fillna({'Service Code': 'UNKNOWN', 'Status': 'UNKNOWN'})

# 2. Impute missing numeric values (replace NULLs with 0)
df_cleaned = df_cleaned.fillna(0, subset=["% AVL to Timetables feed matching score"])

# 3. Final validation of record count
final_count = df_cleaned.count()
print(f"### **Final Cleaned Record Count: {final_count}**")

[Stage 52:=========>        (2 + 2) / 4][Stage 53:=========>        (1 + 1) / 2]

### **Final Cleaned Record Count: 103437**


### 3 - Data Storage & Processing

In [15]:
import sqlite3
import pandas as pd

In [20]:
# 1. Intermediate Write
intermediate_path = os.path.join(DATA_DIR, "cleaned_checkpoint_csv")
df_cleaned.write.mode("overwrite").option("header", "true").csv(intermediate_path)

In [21]:
# 2. Loading Relational Database
db_path = os.path.join(DATA_DIR, "data", "bods_analytics.db")
df_checkpoint = spark.read.option("header", "true").csv(intermediate_path)